TOOLS

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. I remember that parrots are known for their ability to mimic human speech, but why do they do that? Is it just a behavior they\'ve developed through observation, or is there a deeper reason?\n\nFirst, I should consider their natural behavior. In the wild, parrots communicate with each other using a variety of vocalizations. Maybe their ability to mimic humans is an extension of this natural communication skill. But why would they specifically mimic human speech?\n\nI think it has to do with their social nature. Parrots are highly social animals, and in the wild, they rely on vocalizations to stay in contact with their flock. When they\'re in captivity or around humans, they might treat humans as part of their social group. So, they talk to interact, much like how they would with other parrots. That makes sense because it\'s a way for them to communicate and bon

In [2]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])

In [3]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'vk1tsn98v', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.157090978, 'completion_tokens_details': {'reasoning_tokens': 62}, 'prompt_time': 0.014190751, 'prompt_tokens_details': None, 'queue_time': 0.912200531, 'total_time': 0.171281729}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e6a2e-10a2-7e61-9

Tool Execution Loops

In [4]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny right now. Perfect day to enjoy outdoor activities! ☀️


In [5]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. Let me check the function parameters. The required parameter is location, which should be a string. Boston is the location here. So I\'ll call the function with location set to "Boston". Make sure the JSON is correctly formatted with the name and arguments. No other functions are available, so this should be straightforward.\n', 'tool_calls': [{'id': 'dzp6esaa5', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 153, 'total_tokens': 261, 'completion_time': 0.174131542, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.007903327, 'prompt_tokens_details': None, 'queue_time': 0.494387145, 'total_time': 0.182034869}, 'model_name'